# 04 — Baseline: Dummy Classifier y Logistic Regression

Este notebook establece dos referencias antes de comenzar Feature Engineering:

1. un `DummyClassifier`, que representa el piso sin aprendizaje real;
2. una regresión logística sencilla sobre el conjunto original depurado.

El baseline respeta las exclusiones por leakage y disponibilidad temporal acordadas en Feature Analysis, pero no incorpora todavía `TotalGuests`, `TotalNights`, `IsDomestic`, interacciones ni otras features nuevas.

## Estrategia de validación

Se reservan las observaciones de 2017 como holdout temporal y se utilizan 2015–2016 para entrenamiento y selección de hiperparámetros. Como 2017 sólo contiene datos hasta agosto, esta partición mide generalización al período posterior observado, no el desempeño sobre un año calendario completo.

Dentro del conjunto de desarrollo se utiliza `StratifiedGroupKFold`. Los grupos se forman a partir de combinaciones idénticas de predictores, evitando que repeticiones exactas aparezcan a la vez en entrenamiento y validación interna. El holdout de 2017 no se utiliza para seleccionar hiperparámetros.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)

In [ ]:
def find_project_root(start_path=None):
    """Busca la raíz del repositorio a partir del directorio actual."""
    start = Path(start_path or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir() and (candidate / "src").is_dir():
            return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró la raíz local del proyecto.")

project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print(f"Raíz del proyecto: {PROJECT_ROOT}")

In [ ]:
from src.config import RANDOM_STATE, RAW_DATA_DIR, TARGET_COL
from src.data import (
    combine_hotel_datasets,
    load_csv_with_fallback,
    make_temporal_holdout_split,
    normalize_text_values,
)
from src.evaluation import (
    evaluate_binary_classifier,
    get_grid_results,
    get_stratified_group_cross_validation,
    run_grid_search,
)
from src.features import (
    BASE_COLUMNS_TO_EXCLUDE,
    build_base_feature_set,
    get_feature_types,
    make_exact_feature_groups,
)
from src.plots import plot_confusion_matrix
from src.preprocessing import build_tabular_preprocessor

## Carga e integración de los datos

Se priorizan los archivos locales y se utilizan las URLs de GitHub como alternativa. La integración y normalización reutilizan las mismas funciones que el EDA.

In [ ]:
H1_URL = "https://raw.githubusercontent.com/Lithium582/Obligatorio2026/refs/heads/main/data/raw/H1.csv"
H2_URL = "https://raw.githubusercontent.com/Lithium582/Obligatorio2026/refs/heads/main/data/raw/H2.csv"

h1_df = load_csv_with_fallback(RAW_DATA_DIR / "H1.csv", H1_URL)
h2_df = load_csv_with_fallback(RAW_DATA_DIR / "H2.csv", H2_URL)
bookings_df = combine_hotel_datasets(h1_df, h2_df)
bookings_df = normalize_text_values(bookings_df)

print(f"Dataset integrado: {bookings_df.shape[0]:,} filas y {bookings_df.shape[1]} columnas.")

## Conjunto base depurado

Se aplican solamente las decisiones imprescindibles de Feature Analysis: exclusión de variables no disponibles y reemplazo de los identificadores `Agent` y `Company` por `HasAgent` y `HasCompany`. No se crean todavía variables de Feature Engineering.

In [ ]:
X, y = build_base_feature_set(bookings_df)
numeric_features, categorical_features = get_feature_types(X)

feature_summary = pd.DataFrame({
    "group": ["Numeric/binary", "Categorical", "Excluded"],
    "count": [len(numeric_features), len(categorical_features), len(BASE_COLUMNS_TO_EXCLUDE)],
    "columns": [numeric_features, categorical_features, list(BASE_COLUMNS_TO_EXCLUDE)],
})
display(feature_summary)
display(X.isna().sum().loc[lambda values: values.gt(0)].sort_values(ascending=False))

## Holdout temporal y grupos de validación interna

In [ ]:
X_train, X_validation, y_train, y_validation = make_temporal_holdout_split(
    X,
    y,
    period_values=X["ArrivalDateYear"],
    validation_period=2017,
)
train_groups = make_exact_feature_groups(X_train)

split_summary = pd.DataFrame({
    "subset": ["Development (2015-2016)", "Temporal holdout (2017 Jan-Aug)"],
    "rows": [len(X_train), len(X_validation)],
    "cancellation_rate": [y_train.mean(), y_validation.mean()],
    "unique_feature_groups": [train_groups.nunique(), make_exact_feature_groups(X_validation).nunique()],
})
split_summary["cancellation_percentage"] = split_summary["cancellation_rate"] * 100
display(split_summary.round(4))

## Piso de referencia: Dummy Classifier

La estrategia `prior` devuelve la probabilidad observada de cada clase en desarrollo. Su ROC AUC esperado es 0,5; las métricas dependientes del umbral sólo describen su comportamiento con el corte convencional de 0,5.

In [ ]:
dummy_model = DummyClassifier(strategy="prior", random_state=RANDOM_STATE)
dummy_feature = ["ArrivalDateYear"]
dummy_model.fit(X_train[dummy_feature], y_train)
dummy_metrics = evaluate_binary_classifier(
    dummy_model, X_validation[dummy_feature], y_validation
)
display(pd.DataFrame([dummy_metrics], index=["Dummy prior"]).round(4))

## Pipeline de Logistic Regression

Las numéricas se imputan por mediana y se escalan. Las categóricas se imputan por moda y reciben One-Hot Encoding con categorías desconocidas ignoradas. Todo el preprocesamiento se ajusta dentro de cada fold para evitar leakage.

In [ ]:
base_preprocessor = build_tabular_preprocessor(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    scale_numeric=True,
)

logistic_pipeline = Pipeline(steps=[
    ("preprocessor", base_preprocessor),
    ("model", LogisticRegression(solver="liblinear", max_iter=2000, random_state=RANDOM_STATE)),
])

## Búsqueda ligera de hiperparámetros

La grilla es deliberadamente pequeña: el objetivo es obtener una referencia, no optimizar exhaustivamente Logistic Regression antes de Feature Engineering. La selección utiliza exclusivamente los datos de 2015–2016.

In [ ]:
param_grid = {
    "model__C": [0.1, 1.0, 10.0],
    "model__class_weight": [None, "balanced"],
}
group_cv = get_stratified_group_cross_validation(n_splits=5)

logistic_search = run_grid_search(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    X=X_train,
    y=y_train,
    scoring="roc_auc",
    cv=group_cv,
    groups=train_groups,
    n_jobs=-1,
)
grid_results = get_grid_results(logistic_search)
display(grid_results[["params", "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]].round(4))
print(f"Mejores parámetros: {logistic_search.best_params_}")
print(f"Mejor ROC AUC CV: {logistic_search.best_score_:.4f}")

## Evaluación única sobre el holdout temporal

El mejor pipeline según validación interna se evalúa una sola vez sobre 2017. ROC AUC es la métrica principal. Accuracy, precision, recall y F1 se muestran con umbral 0,5 como información secundaria; el ajuste del umbral se realizará más adelante con el análisis de negocio.

In [ ]:
best_logistic_model = logistic_search.best_estimator_
logistic_metrics = evaluate_binary_classifier(
    best_logistic_model, X_validation, y_validation, threshold=0.5
)

baseline_comparison = pd.DataFrame([
    {"model": "Dummy prior", **dummy_metrics},
    {"model": "Logistic Regression base", **logistic_metrics},
]).set_index("model")
display(baseline_comparison.round(4))

In [ ]:
plot_confusion_matrix(
    best_logistic_model,
    X_validation,
    y_validation,
    threshold=0.5,
    title="Logistic Regression base — holdout temporal 2017",
)

## Interpretación pendiente tras la ejecución

Después de ejecutar el notebook se registrarán:

- la mejora de ROC AUC de Logistic Regression respecto al dummy;
- la diferencia entre validación cruzada agrupada y holdout temporal;
- la estabilidad de los resultados entre folds;
- los hiperparámetros seleccionados;
- y las limitaciones observadas con el umbral convencional.

Este resultado será la referencia para medir el aporte posterior de Feature Engineering y de modelos más flexibles.